# Hybrid Model Pipeline
## Graph + Text Feature Engineering for Fake Review Detection

**Pipeline Steps:**
1. Load preprocessed data (nodes, edges, features, embeddings)
2. Build graph structure for GNN
3. Combine text embeddings with structured features
4. Prepare training data (graph + text + weak labels)
5. Create data loaders for model training

## 1. Setup & Imports

In [1]:
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import json
from collections import defaultdict
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# === SETUP PATHS ===
PROJECT_DIR = Path(".")
# .resolve() giúp in ra đường dẫn tuyệt đối thực tế để bạn quan sát
print(f"Data directory: {PROJECT_DIR.resolve()}")
print(f"\nFiles available:")

GOLD_DIR = PROJECT_DIR / "data" / "gold"
CATEGORY = "All_Beauty"
DATA_DIR = GOLD_DIR / CATEGORY

print(f"Project Directory: {PROJECT_DIR}")
print(f"Data Directory: {DATA_DIR}")

# Create output directory for hybrid model data
HYBRID_DIR = DATA_DIR / "hybrid_model"
HYBRID_DIR.mkdir(parents=True, exist_ok=True)
print(f"Hybrid Model Directory: {HYBRID_DIR}")

Data directory: /Users/mac/Downloads/MXH FINAL

Files available:
Project Directory: .
Data Directory: data/gold/All_Beauty
Hybrid Model Directory: data/gold/All_Beauty/hybrid_model


## 2. Load Preprocessed Data

In [3]:
# === LOAD NODES ===
logger.info("Loading node tables...")
nodes_user = pd.read_parquet(DATA_DIR / "nodes_user.parquet")
nodes_review = pd.read_parquet(DATA_DIR / "nodes_review.parquet")
nodes_item = pd.read_parquet(DATA_DIR / "nodes_item.parquet")

print(f"✓ Users: {len(nodes_user)}")
print(f"✓ Reviews: {len(nodes_review)}")
print(f"✓ Items: {len(nodes_item)}")

2026-06-10 22:56:42,797 - INFO - Loading node tables...


✓ Users: 588332
✓ Reviews: 644689
✓ Items: 108784


In [4]:
# === LOAD EDGES ===
logger.info("Loading edge tables...")
edges_user_review = pd.read_parquet(DATA_DIR / "edges_user_review.parquet")
edges_review_item = pd.read_parquet(DATA_DIR / "edges_review_item.parquet")
edges_user_item = pd.read_parquet(DATA_DIR / "edges_user_item.parquet")
edges_review_review = pd.read_parquet(DATA_DIR / "edges_review_review.parquet")

print(f"✓ User-Review edges: {len(edges_user_review)}")
print(f"✓ Review-Item edges: {len(edges_review_item)}")
print(f"✓ User-Item edges: {len(edges_user_item)}")
print(f"✓ Review-Review edges: {len(edges_review_review)}")

2026-06-10 22:56:44,395 - INFO - Loading edge tables...


✓ User-Review edges: 644689
✓ Review-Item edges: 644689
✓ User-Item edges: 644689
✓ Review-Review edges: 267837


In [5]:
# === CELL 4: LOAD FEATURES & LABELS RAW ===
logger.info("Loading features and labels...")
review_features = pd.read_parquet(DATA_DIR / "review_features.parquet")
review_splits = pd.read_parquet(DATA_DIR / "review_splits.parquet")

print(f"✓ Review features: {review_features.shape}")
print(f"✓ Review splits: {review_splits.shape}")
print(f"\nSplit distribution:")
print(review_splits["split"].value_counts())
print(f"\nWeak label distribution:")
print(review_splits["weak_label"].value_counts())

2026-06-10 22:56:45,165 - INFO - Loading features and labels...


✓ Review features: (644689, 39)
✓ Review splits: (644689, 4)

Split distribution:
split
train    451282
test      96704
val       96703
Name: count, dtype: int64

Weak label distribution:
weak_label
0    576231
1     68458
Name: count, dtype: int64


In [6]:
# === LOAD TEXT EMBEDDINGS ===
logger.info("Loading text embeddings...")
embedding_path = DATA_DIR / f"review_text_embeddings_{CATEGORY}.npy"

if embedding_path.exists():
    review_embeddings = np.load(embedding_path)
    print(f"✓ Embeddings shape: {review_embeddings.shape}")
    print(f"  - Reviews: {review_embeddings.shape[0]}")
    print(f"  - Embedding dimension: {review_embeddings.shape[1]}")
else:
    print("⚠ No embeddings found - will create dummy embeddings")
    review_embeddings = None

2026-06-10 22:56:45,574 - INFO - Loading text embeddings...


✓ Embeddings shape: (644689, 384)
  - Reviews: 644689
  - Embedding dimension: 384


## 3. Build Graph Structure

In [7]:
class GraphBuilder:
    """Build heterogeneous graph for GNN."""
    
    def __init__(self, nodes_user, nodes_review, nodes_item, edges):
        self.nodes_user = nodes_user
        self.nodes_review = nodes_review
        self.nodes_item = nodes_item
        
        # Create ID mappings
        self.user_id_map = {uid: i for i, uid in enumerate(nodes_user['user_id'])}
        self.review_id_map = {rid: i for i, rid in enumerate(nodes_review['review_id'])}
        self.item_id_map = {iid: i for i, iid in enumerate(nodes_item['item_id'])}
        
        self.edges = edges
        logger.info(f"Graph built with {len(self.user_id_map)} users, {len(self.review_id_map)} reviews, {len(self.item_id_map)} items")
    
    def get_adjacency_matrix(self, edge_type: str) -> Tuple[List[Tuple], str]:
        """
        Get edge list and node types for specific edge type.
        """
        edge_data = self.edges[edge_type]
        edges_list = []
        
        if edge_type == 'edges_user_review':
            for _, row in edge_data.iterrows():
                if row['src'] in self.user_id_map and row['dst'] in self.review_id_map:
                    edges_list.append((self.user_id_map[row['src']], self.review_id_map[row['dst']]))
            node_types = ("user", "review")
            
        elif edge_type == 'edges_review_item':
            for _, row in edge_data.iterrows():
                if row['src'] in self.review_id_map and row['dst'] in self.item_id_map:
                    edges_list.append((self.review_id_map[row['src']], self.item_id_map[row['dst']]))
            node_types = ("review", "item")
            
        elif edge_type == 'edges_user_item':
            for _, row in edge_data.iterrows():
                if row['src'] in self.user_id_map and row['dst'] in self.item_id_map:
                    edges_list.append((self.user_id_map[row['src']], self.item_id_map[row['dst']]))
            node_types = ("user", "item")
            
        elif edge_type == 'edges_review_review':
            for _, row in edge_data.iterrows():
                if row['src'] in self.review_id_map and row['dst'] in self.review_id_map:
                    edges_list.append((self.review_id_map[row['src']], self.review_id_map[row['dst']]))
            node_types = ("review", "review")
        
        return edges_list, node_types
    
    def get_graph_dict(self) -> Dict:
        """
        Get complete graph as dictionary with all edge types.
        """
        graph = {}
        for edge_type_key in self.edges.keys():
            edges_list, node_types = self.get_adjacency_matrix(edge_type_key)
            if len(edges_list) > 0:
                graph[edge_type_key] = {
                    'edges': edges_list,
                    'node_types': node_types,
                    'num_edges': len(edges_list)
                }
        return graph

# Build graph
edges_dict = {
    'edges_user_review': edges_user_review,
    'edges_review_item': edges_review_item,
    'edges_user_item': edges_user_item,
    'edges_review_review': edges_review_review
}

# Khởi tạo GraphBuilder với danh sách nút đã đồng bộ hóa
graph_builder = GraphBuilder(nodes_user, nodes_review, nodes_item, edges_dict)
graph_dict = graph_builder.get_graph_dict()

logger.info(f"Graph structure built with {len(graph_dict)} edge types")
for edge_type, data in graph_dict.items():
    print(f"  {edge_type}: {data['num_edges']} edges ({data['node_types']})")

2026-06-10 22:56:46,486 - INFO - Graph built with 588332 users, 644689 reviews, 108784 items
2026-06-10 22:57:43,502 - INFO - Graph structure built with 4 edge types


  edges_user_review: 644689 edges (('user', 'review'))
  edges_review_item: 644689 edges (('review', 'item'))
  edges_user_item: 644689 edges (('user', 'item'))
  edges_review_review: 267837 edges (('review', 'review'))


## 4. Prepare Hybrid Features (Graph + Text)

In [8]:
# === CELL 7: HYBRID FEATURE BUILDER WITH ZERO-PADDING OPTIMIZATION ===
class HybridFeatureBuilder:
    """Combine graph structural features with text embeddings using zero-padding for alignment."""
    
    def __init__(self, review_features, review_embeddings, nodes_review, review_splits):
        self.review_features = review_features
        self.review_embeddings = review_embeddings
        self.nodes_review = nodes_review
        self.review_splits = review_splits
        self.total_nodes = len(nodes_review)  # Đảm bảo giữ đúng 693,929 nút
        
    def get_structural_features(self) -> np.ndarray:
        """Extract structured features from review_features."""
        exclude_cols = ['review_id', 'weak_label', 'suspicion_score']
        feature_cols = [col for col in self.review_features.columns if col not in exclude_cols]
        
        structural_features = self.review_features[feature_cols].values.astype(np.float32)
        structural_features = np.nan_to_num(structural_features, nan=0.0, posinf=0.0, neginf=0.0)
        
        logger.info(f"Structural features shape: {structural_features.shape}")
        return structural_features, feature_cols
    
    def get_text_embeddings(self) -> np.ndarray:
        """Get text embeddings and pad missing rows with zeros."""
        # Khởi tạo ma trận zero toàn phần kích thước (693929, 384)
        text_emb = np.zeros((self.total_nodes, 384), dtype=np.float32)
        
        if self.review_embeddings is not None:
            # Điền các vector nhúng văn bản đang có vào phần đầu ma trận
            available_len = min(len(self.review_embeddings), self.total_nodes)
            text_emb[:available_len] = self.review_embeddings[:available_len]
            
            # Chuẩn hóa L2-norm cho các hàng chứa dữ liệu thực tế
            norms = np.linalg.norm(text_emb, axis=1, keepdims=True)
            norms[norms == 0] = 1
            text_emb = text_emb / norms
        else:
            logger.warning("No embeddings available - creating dummy embeddings")
            text_emb = np.random.randn(self.total_nodes, 384).astype(np.float32)
            
        logger.info(f"Text embeddings shape (aligned with padding): {text_emb.shape}")
        return text_emb
    
    def combine_features(self, structural_features, text_embeddings) -> np.ndarray:
        """Combine structural features and text embeddings (NO anomaly scores for clean baseline)."""
        # ⚠️ REMOVED: Anomaly scores (hybrid_anomaly_scores, graph_suspicion, semantic_suspicion)
        # These are kept separate for the hybrid model to prevent data leakage
        # XGBoost baseline uses text_embeddings ONLY
        # Hybrid model uses both graph structure + combined features
        
        combined = np.concatenate([structural_features, text_embeddings], axis=1)
        logger.info(f"Combined features shape: {combined.shape}")
        logger.info(f"  - Structural: {structural_features.shape[1]} dims")
        logger.info(f"  - Text: {text_embeddings.shape[1]} dims")
        return combined
    
    def get_labels_and_splits(self):
        """Get weak labels and train/val/test splits."""
        labels = self.review_splits['weak_label'].values.astype(np.int32)
        split_map = {'train': 0, 'val': 1, 'test': 2}
        splits = self.review_splits['split'].map(split_map).values.astype(np.int32)
        
        return labels, splits

# Tiến hành khởi tạo và xử lý trích xuất đặc trưng lai lai
feature_builder = HybridFeatureBuilder(review_features, review_embeddings, nodes_review, review_splits)

structural_features, feature_cols = feature_builder.get_structural_features()
text_embeddings = feature_builder.get_text_embeddings()
combined_features = feature_builder.combine_features(structural_features, text_embeddings)
labels, splits = feature_builder.get_labels_and_splits()

print(f"\n✓ Hybrid features ready: {combined_features.shape}")

2026-06-10 22:57:43,884 - INFO - Structural features shape: (644689, 36)
2026-06-10 22:57:45,986 - INFO - Text embeddings shape (aligned with padding): (644689, 384)
2026-06-10 22:57:46,735 - INFO - Combined features shape: (644689, 420)
2026-06-10 22:57:46,736 - INFO -   - Structural: 36 dims
2026-06-10 22:57:46,736 - INFO -   - Text: 384 dims



✓ Hybrid features ready: (644689, 420)


## 5. Create Training Data Package

In [9]:
class TrainingDataPackage:
    """Package all data needed for model training."""
    
    def __init__(self, graph_dict, combined_features, labels, splits, nodes_review, review_id_map):
        self.graph_dict = graph_dict
        self.combined_features = combined_features
        self.labels = labels
        self.splits = splits
        self.nodes_review = nodes_review
        self.review_id_map = review_id_map
    
    def get_split_indices(self):
        """
        Get indices for train/val/test splits.
        """
        train_idx = np.where(self.splits == 0)[0]
        val_idx = np.where(self.splits == 1)[0]
        test_idx = np.where(self.splits == 2)[0]
        
        return {
            'train': train_idx,
            'val': val_idx,
            'test': test_idx
        }
    
    def summary(self):
        """
        Print training data summary.
        """
        split_indices = self.get_split_indices()
        
        print("\n" + "="*60)
        print("TRAINING DATA PACKAGE SUMMARY")
        print("="*60)
        print(f"\nFeatures:")
        print(f"  Total reviews: {len(self.nodes_review)}")
        print(f"  Combined feature dimension: {self.combined_features.shape[1]}")
        print(f"\nGraph:")
        for edge_type, data in self.graph_dict.items():
            print(f"  {edge_type}: {data['num_edges']} edges")
        print(f"\nLabels (Weak Label Distribution):")
        unique, counts = np.unique(self.labels, return_counts=True)
        for label, count in zip(unique, counts):
            pct = count / len(self.labels) * 100
            print(f"  Label {label}: {count:,} ({pct:.1f}%)")
        print(f"\nData Split:")
        print(f"  Train: {len(split_indices['train']):,} ({len(split_indices['train'])/len(self.labels)*100:.1f}%)")
        print(f"  Val:   {len(split_indices['val']):,} ({len(split_indices['val'])/len(self.labels)*100:.1f}%)")
        print(f"  Test:  {len(split_indices['test']):,} ({len(split_indices['test'])/len(self.labels)*100:.1f}%)")
        print("="*60)

# Create training data package
training_pkg = TrainingDataPackage(
    graph_dict,
    combined_features,
    labels,
    splits,
    nodes_review,
    graph_builder.review_id_map
)

training_pkg.summary()


TRAINING DATA PACKAGE SUMMARY

Features:
  Total reviews: 644689
  Combined feature dimension: 420

Graph:
  edges_user_review: 644689 edges
  edges_review_item: 644689 edges
  edges_user_item: 644689 edges
  edges_review_review: 267837 edges

Labels (Weak Label Distribution):
  Label 0: 576,231 (89.4%)
  Label 1: 68,458 (10.6%)

Data Split:
  Train: 451,282 (70.0%)
  Val:   96,703 (15.0%)
  Test:  96,704 (15.0%)


## 6. Save Training Data

In [10]:
# === SAVE FEATURES ===
logger.info(f"Saving features to {HYBRID_DIR}...")

np.save(HYBRID_DIR / "combined_features.npy", combined_features)
np.save(HYBRID_DIR / "text_embeddings_aligned.npy", text_embeddings)
np.save(HYBRID_DIR / "structural_features.npy", structural_features)
np.save(HYBRID_DIR / "labels.npy", labels)
np.save(HYBRID_DIR / "splits.npy", splits)

logger.info("✓ Features saved")
logger.info(f"  - combined_features.npy: {combined_features.shape}")
logger.info(f"  - text_embeddings_aligned.npy: {text_embeddings.shape}")
logger.info(f"  - structural_features.npy: {structural_features.shape}")

2026-06-10 22:57:46,862 - INFO - Saving features to data/gold/All_Beauty/hybrid_model...
2026-06-10 22:57:47,950 - INFO - ✓ Features saved
2026-06-10 22:57:47,951 - INFO -   - combined_features.npy: (644689, 420)
2026-06-10 22:57:47,951 - INFO -   - text_embeddings_aligned.npy: (644689, 384)
2026-06-10 22:57:47,952 - INFO -   - structural_features.npy: (644689, 36)


In [11]:
# === SAVE GRAPH ===
logger.info("Saving graph structure...")

graph_json = {}
for edge_type, data in graph_dict.items():
    graph_json[edge_type] = {
        'edges': [[int(u), int(v)] for u, v in data['edges']],
        'node_types': data['node_types'],
        'num_edges': data['num_edges']
    }

with open(HYBRID_DIR / "graph_structure.json", "w") as f:
    json.dump(graph_json, f, indent=2)

logger.info("✓ Graph saved")

2026-06-10 22:57:47,958 - INFO - Saving graph structure...
2026-06-10 22:57:54,139 - INFO - ✓ Graph saved


In [12]:
# === SAVE METADATA ===
logger.info("Saving metadata...")

metadata = {
    'num_reviews': len(nodes_review),
    'num_users': len(nodes_user),
    'num_items': len(nodes_item),
    'feature_dimension': combined_features.shape[1],
    'structural_features': structural_features.shape[1],
    'text_embedding_dim': text_embeddings.shape[1],
    'note': 'Combined features = structural_features + text_embeddings (NO anomaly scores)',
    'num_edge_types': len(graph_dict),
    'total_edges': sum(data['num_edges'] for data in graph_dict.values()),
    'label_distribution': {
        str(int(k)): int(v) for k, v in zip(*np.unique(labels, return_counts=True))
    },
    'split_distribution': {
        'train': int(np.sum(splits == 0)),
        'val': int(np.sum(splits == 1)),
        'test': int(np.sum(splits == 2))
    }
}

with open(HYBRID_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

logger.info("✓ Metadata saved")

2026-06-10 22:57:54,181 - INFO - Saving metadata...
2026-06-10 22:57:54,192 - INFO - ✓ Metadata saved


In [13]:
# === SAVE NODE MAPPINGS ===
logger.info("Saving ID mappings...")

mappings = {
    'review_id_map': {k: v for k, v in graph_builder.review_id_map.items()},
    'user_id_map': {k: v for k, v in graph_builder.user_id_map.items()},
    'item_id_map': {k: v for k, v in graph_builder.item_id_map.items()}
}

with open(HYBRID_DIR / "id_mappings.json", "w") as f:
    json.dump(mappings, f)

logger.info("✓ ID mappings saved")

2026-06-10 22:57:54,197 - INFO - Saving ID mappings...


2026-06-10 22:57:55,465 - INFO - ✓ ID mappings saved


## 7. Verification & Summary

In [14]:
# === VERIFY SAVED FILES ===
print("\nSaved files:")
for file in sorted(HYBRID_DIR.iterdir()):
    if file.is_file():
        if file.suffix == '.npy':
            size = file.stat().st_size / (1024 * 1024)
            arr = np.load(file)
            print(f"  ✓ {file.name:<35} {arr.shape} ({size:.2f} MB)")
        elif file.suffix == '.json':
            size = file.stat().st_size / 1024
            print(f"  ✓ {file.name:<35} ({size:.2f} KB)")


Saved files:
  ✓ combined_features.npy               (644689, 420) (1032.90 MB)
  ✓ graph_structure.json                (101158.36 KB)
  ✓ id_mappings.json                    (52694.99 KB)
  ✓ labels.npy                          (644689,) (2.46 MB)
  ✓ metadata.json                       (0.44 KB)
  ✓ splits.npy                          (644689,) (2.46 MB)
  ✓ structural_features.npy             (644689, 36) (88.53 MB)
  ✓ text_embeddings_aligned.npy         (644689, 384) (944.37 MB)


In [15]:
# === FINAL SUMMARY ===
print("\n" + "="*70)
print("HYBRID MODEL PIPELINE COMPLETE")
print("="*70)
print(f"\nOutput Directory: {HYBRID_DIR}")
print(f"\nDataset Statistics:")
print(f"  Total Samples: {len(labels):,}")
print(f"  Feature Dimension: {combined_features.shape[1]}")
print(f"    - Structural Features: {structural_features.shape[1]}")
print(f"    - Text Embeddings: {text_embeddings.shape[1]}")
print(f"\nGraph Statistics:")
for edge_type, data in graph_dict.items():
    print(f"  {edge_type}: {data['num_edges']:,}")
print(f"\nClass Distribution:")
unique_labels, label_counts = np.unique(labels, return_counts=True)
for label, count in zip(unique_labels, label_counts):
    pct = count / len(labels) * 100
    print(f"  Class {label}: {count:,} ({pct:.1f}%)")
print(f"\nData Splits:")
print(f"  Train: {np.sum(splits == 0):,} samples")
print(f"  Val:   {np.sum(splits == 1):,} samples")
print(f"  Test:  {np.sum(splits == 2):,} samples")
print("\nReady for Model Training! 🚀")
print("="*70)


HYBRID MODEL PIPELINE COMPLETE

Output Directory: data/gold/All_Beauty/hybrid_model

Dataset Statistics:
  Total Samples: 644,689
  Feature Dimension: 420
    - Structural Features: 36
    - Text Embeddings: 384

Graph Statistics:
  edges_user_review: 644,689
  edges_review_item: 644,689
  edges_user_item: 644,689
  edges_review_review: 267,837

Class Distribution:
  Class 0: 576,231 (89.4%)
  Class 1: 68,458 (10.6%)

Data Splits:
  Train: 451,282 samples
  Val:   96,703 samples
  Test:  96,704 samples

Ready for Model Training! 🚀


## 8. Data Loading Example

In [16]:
# === EXAMPLE: HOW TO LOAD DATA FOR TRAINING ===
print("Example code to load data for training:\n")
print("""
# Load hybrid model data
from pathlib import Path
import numpy as np
import json

HYBRID_DIR = Path("/Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model")

# Load features
features = np.load(HYBRID_DIR / "combined_features.npy")  # (N, D)
labels = np.load(HYBRID_DIR / "labels.npy")  # (N,)
splits = np.load(HYBRID_DIR / "splits.npy")  # (N,) - 0:train, 1:val, 2:test

# Load graph
with open(HYBRID_DIR / "graph_structure.json") as f:
    graph = json.load(f)

# Load metadata
with open(HYBRID_DIR / "metadata.json") as f:
    metadata = json.load(f)

# Get train/val/test indices
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

# Create data loaders (PyTorch example)
from torch.utils.data import TensorDataset, DataLoader
import torch

X_train = torch.FloatTensor(features[train_idx])
y_train = torch.LongTensor(labels[train_idx])
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
""")

Example code to load data for training:


# Load hybrid model data
from pathlib import Path
import numpy as np
import json

HYBRID_DIR = Path("/Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model")

# Load features
features = np.load(HYBRID_DIR / "combined_features.npy")  # (N, D)
labels = np.load(HYBRID_DIR / "labels.npy")  # (N,)
splits = np.load(HYBRID_DIR / "splits.npy")  # (N,) - 0:train, 1:val, 2:test

# Load graph
with open(HYBRID_DIR / "graph_structure.json") as f:
    graph = json.load(f)

# Load metadata
with open(HYBRID_DIR / "metadata.json") as f:
    metadata = json.load(f)

# Get train/val/test indices
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

# Create data loaders (PyTorch example)
from torch.utils.data import TensorDataset, DataLoader
import torch

X_train = torch.FloatTensor(features[train_idx])
y_train = torch.LongTensor(labels[train_idx])
train_dataset = TensorDataset(X_train, y_train)
tra